### Fake News Detection on LIAR Dataset using SBERT Embeddings

This notebook performs fake news classification on the LIAR dataset
using semantic embeddings generated by Sentence-BERT (SBERT).

The embeddings are used as input features for a Logistic Regression
classifier to detect whether a political claim is real or fake.

#### Objective

The goal of this experiment is to evaluate the effectiveness of
Sentence-BERT embeddings for fake news detection on short political
statements from the LIAR dataset.

Pipeline:

Text → spaCy Preprocessing → SBERT Embeddings → Logistic Regression

#### Dataset

Dataset Used:
LIAR Fake News Dataset

The dataset contains short political statements labeled as real
or fake.

Class Labels:

0 → Fake claim
1 → Real claim

Test Samples: 790

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('train2.tsv',sep='\t',header=None)
test_df = pd.read_csv('test2.tsv',sep='\t',header=None)
print(df.shape)
df.head()

(10242, 16)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,0.0,2635.json,false,Says the Annies List political group supports ...,abortion,dwayne-bohac,State representative,Texas,republican,0.0,1.0,0.0,0.0,0.0,a mailer,That's a premise that he fails to back up. Ann...
1,1.0,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0.0,0.0,1.0,1.0,0.0,a floor speech.,"Surovell said the decline of coal ""started whe..."
2,2.0,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70.0,71.0,160.0,163.0,9.0,Denver,Obama said he would have voted against the ame...
3,3.0,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting,NaN,NaN,none,7.0,19.0,3.0,5.0,44.0,a news release,The release may have a point that Mikulskis co...
4,4.0,9028.json,half-true,The economic turnaround started at the end of ...,"economy,jobs",charlie-crist,NaN,Florida,democrat,15.0,9.0,20.0,19.0,2.0,an interview on CNN,"Crist said that the economic ""turnaround start..."


In [3]:
df = df[[2,3]]
df.columns = ['label','text']
df.head()

,label,text
0,false,Says the Annies List political group supports ...
1,half-true,When did the decline of coal start? It started...
2,mostly-true,"Hillary Clinton agrees with John McCain ""by vo..."
3,false,Health care reform legislation is likely to ma...
4,half-true,The economic turnaround started at the end of ...


In [4]:
test_df = test_df[[2,3]]
test_df.columns = ['label','text']
test_df.head()

,label,text
0,true,Building a wall on the U.S.-Mexico border will...
1,false,Wisconsin is on pace to double the number of l...
2,false,Says John McCain has done nothing to help the ...
3,half-true,Suzanne Bonamici supports a plan that will cut...
4,pants-fire,When asked by a reporter whether hes at the ce...


In [5]:
df.isnull().sum()

label    2
text     2
dtype: int64

In [6]:
test_df.isnull().sum()

label    0
text     0
dtype: int64

In [7]:
df=df.dropna()

In [8]:
df.isnull().sum()

label    0
text     0
dtype: int64

In [9]:
df['label'].value_counts()

label
half-true      2114
false          1995
mostly-true    1962
true           1676
barely-true    1654
pants-fire      839
Name: count, dtype: int64

In [10]:
test_df['label'].value_counts()

label
half-true      265
false          249
mostly-true    241
barely-true    212
true           208
pants-fire      92
Name: count, dtype: int64

In [11]:
df = df[df['label'].isin(['false','pants-fire','true','mostly-true'])]
df['label_num'] = df['label'].apply(lambda x:1 if x in ['true','mostly-true'] else 0)

In [12]:
test_df = test_df[test_df['label'].isin(['false','pants-fire','true','mostly-true'])]
test_df['label_num'] = test_df['label'].apply(lambda x:1 if x in ['true','mostly-true'] else 0)

In [13]:
test_df.head()

,label,text,label_num
0,true,Building a wall on the U.S.-Mexico border will...,1
1,false,Wisconsin is on pace to double the number of l...,0
2,false,Says John McCain has done nothing to help the ...,0
4,pants-fire,When asked by a reporter whether hes at the ce...,0
5,true,Over the past five years the federal governmen...,1


#### Text Preprocessing

Text preprocessing is performed using spaCy.

The following steps are applied:

• Remove punctuation
• Remove stopwords
• Lemmatization

spaCy helps normalize the text and reduce noise before generating
sentence embeddings.

In [14]:
import spacy
nlp = spacy.load('en_core_web_sm')
def preprocessing(text):
    doc = nlp(text)
    filtered_tokens = []
    for token in doc:
        if token.is_stop or token.is_punct:
            continue
        filtered_tokens.append(token.lemma_)
    return ' '.join(filtered_tokens)

In [15]:
df['processed_text'] = df['text'].apply(preprocessing)

In [16]:
test_df['processed_text'] = test_df['text'].apply(preprocessing)

In [17]:
test_df.head()

,label,text,label_num,processed_text
0,true,Building a wall on the U.S.-Mexico border will...,1,build wall U.S.-Mexico border literally year
1,false,Wisconsin is on pace to double the number of l...,0,Wisconsin pace double number layoff year
2,false,Says John McCain has done nothing to help the ...,0,say John McCain help vet
4,pants-fire,When asked by a reporter whether hes at the ce...,0,ask reporter s center criminal scheme violate ...
5,true,Over the past five years the federal governmen...,1,past year federal government pay $ 601 million...


In [18]:
df.head()

,label,text,label_num,processed_text
0,false,Says the Annies List political group supports ...,0,say Annies List political group support trimes...
2,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",1,Hillary Clinton agree John McCain vote George ...
3,false,Health care reform legislation is likely to ma...,0,health care reform legislation likely mandate ...
5,true,The Chicago Bears have had more starting quart...,1,Chicago Bears starting quarterback 10 year tot...
9,mostly-true,Says GOP primary opponents Glenn Grothman and ...,1,say GOP primary opponent Glenn Grothman Joe Le...


In [19]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import numpy as np

#### Sentence Embeddings using SBERT

Sentence-BERT is used to generate semantic embeddings for each claim.

Model Used:
all-MiniLM-L6-v2

Each statement is converted into a dense vector representation
that captures contextual meaning rather than simple word frequency.

These embeddings are then used as input features for the classifier.

In [20]:
model = SentenceTransformer('all-MiniLM-L6-v2')

X_train_emb = model.encode(df['processed_text'].tolist())
X_test_emb = model.encode(test_df['processed_text'].tolist())

## Classification Model

Classifier:
Logistic Regression

The model is trained on SBERT embeddings to classify political
claims as real or fake.

In [21]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_emb, df['label_num'])

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [22]:
y_pred = clf.predict(X_test_emb)

In [23]:
print("Accuracy:", accuracy_score(test_df['label_num'], y_pred))
print("Macro F1:", f1_score(test_df['label_num'], y_pred, average='macro'))
print("Weighted F1:", f1_score(test_df['label_num'], y_pred, average='weighted'))
print(confusion_matrix(test_df['label_num'], y_pred))
print(classification_report(test_df['label_num'], y_pred))

Accuracy: 0.6544303797468355
Macro F1: 0.633450548031116
Weighted F1: 0.6454390232972415
[[164 177]
 [ 96 353]]
              precision    recall  f1-score   support

           0       0.63      0.48      0.55       341
           1       0.67      0.79      0.72       449

    accuracy                           0.65       790
   macro avg       0.65      0.63      0.63       790
weighted avg       0.65      0.65      0.65       790



#### Results
Accuracy: 0.654

Macro F1: 0.633
Weighted F1: 0.645

Confusion Matrix

[[164 177]
 [ 96 353]]

#### Observations

The model achieves approximately 65% accuracy on the LIAR dataset.

Compared to TF-IDF based models (~63–64% accuracy), SBERT embeddings
provide a slight improvement.

This improvement occurs because SBERT captures semantic relationships
between words and phrases, which is important when dealing with short
political claims.

However, the model still shows class imbalance behavior, predicting
real claims more accurately than fake claims.

Further experiments are conducted to evaluate cross-domain
performance using SBERT embeddings.